# 课后练习解答（02.04_model_structure）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 深度可分离卷积中，Depthwise 卷积的 groups 应设置为？
A. 1
B. 输入通道数
C. 输出通道数
D. batch size

**解答：** B

**解析：** Depthwise 对每个输入通道独立做空间卷积，因此 groups 必须等于输入通道数。


### 问题2（单选题）

**题目：** MobileBlock 中 expansion ratio=4、输入通道 C_in=16 时，Depthwise 卷积的输入通道数是？
A. 16
B. 64
C. 4
D. 224

**解答：** B

**解析：** 先经 1×1 扩展层把通道升到 16×4=64，Depthwise 在 64 个通道上逐通道卷积。


### 问题3（单选题）

**题目：** h-swish 的表达式为 x·ReLU6(x+3)/6。当 x=6 时，输出为？
A. 0
B. 3
C. 6
D. 9

**解答：** C

**解析：** ReLU6(9)=6，因此输出为 6×6/6=6。


### 问题4（多选题）

**题目：** MobileNetV3 降低计算量的设计包括？
A. 深度可分离卷积
B. 1×1 瓶颈投影
C. width multiplier
D. 使用更大的 7×7 卷积核

**解答：** ABC

**解析：** 更大的卷积核会增加计算量，与轻量化目标相反。


### 问题5（多选题）

**题目：** SE 模块（Squeeze-and-Excitation）包含哪些操作？
A. 全局平均池化
B. 降维 FC + ReLU
C. 升维 FC + 激活
D. 对特征图逐通道缩放

**解答：** ABCD

**解析：** Squeeze 用全局平均池化压缩空间，Excitation 用两个 FC 学习通道权重，最后 scale 回特征图。


### 问题6（判断题）

**题目：** Squeeze 步骤使用全局平均池化将每个通道压缩为单个标量。

**解答：** 对

**解析：** 全局平均池化把 H×W 特征压缩为 1×1×C，得到通道描述向量。


### 问题7（判断题）

**题目：** h-swish 内部包含 sigmoid 与指数运算，部署成本与原始 swish 完全相同。

**解答：** 错

**解析：** h-swish 使用 ReLU6 构造，是分段线性近似，不包含 sigmoid 与指数运算。


### 问题8（填空题）

**题目：** MobileNetV3-Large 中 MobileBlock 的数量为 ____。

**解答：** 15


### 问题9（填空题）

**题目：** width multiplier=0.75 时，通道数约为原来的 0.75 倍，参数量理论上约为原来的 ____ 倍。

**解答：** 0.75^2≈0.5625


### 问题10（简答题）

**题目：** 为什么 MobileBlock 只有在 stride=1 且输入输出通道相同时才使用残差连接？

**解答：** 残差连接要求输入与输出逐元素相加，shape 必须完全一致；stride=1 且通道相同时才满足恒等映射条件，否则需要额外的 1×1 投影或下采样。


### 问题11（简答题）

**题目：** 为什么 SE 模块通常放在 Depthwise 之后而不是 Pointwise 之前？结合感受野和通道注意力解释。

**解答：** Depthwise 之后特征图已聚合局部空间信息，此时全局平均池化得到的通道描述更接近真实语义；若放在 Pointwise 之前，通道尚未完成跨通道融合，SE 学到的权重会更偏向局部噪声。


### 问题12（代码设计题）

**题目：** 补全 SqueezeBlock.forward：输入 x，先全局平均池化，再两个 FC 得到通道权重，最后与 x 逐通道相乘。

**解答：** ```python
import torch.nn as nn

class SqueezeBlock(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        hidden = max(in_channels // reduction, 8)
        self.fc1 = nn.Linear(in_channels, hidden)
        self.fc2 = nn.Linear(hidden, in_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        b, c, h, w = x.shape
        y = x.mean(dim=(2, 3), keepdim=True)  # 全局平均池化
        y = y.view(b, c)
        y = self.fc2(self.relu(self.fc1(y)))  # 两个 FC
        y = torch.sigmoid(y).view(b, c, 1, 1)
        return x * y
```


### 问题13（单选题）

**题目：** MobileBlock 中真正完成跨通道信息混合的是？
A. 3×3 Depthwise
B. 1×1 Pointwise
C. BN
D. h-swish

**解答：** B

**解析：** Depthwise 只做空间卷积，Pointwise 1×1 卷积负责跨通道线性组合。


### 问题14（多选题）

**题目：** 关于深度可分离卷积的正确结论包括？
A. 空间卷积与通道混合被解耦
B. Depthwise groups=输入通道数
C. Pointwise 负责通道混合
D. 相同通道数下 FLOPs 通常低于标准卷积

**解答：** ABCD

**解析：** 四个描述均成立，这正是 MobileNet 系列轻量化的核心。


### 问题15（简答题）

**题目：** 设 C=64、K=3、H=W=112，分别写出标准卷积和 Depthwise+Pointwise 的 FLOPs 表达式，并计算比例。

**解答：** 标准卷积 FLOPs = C×C×K×K×H×W = 64×64×9×112×112。深度可分离 = Depthwise C×K×K×H×W + Pointwise C×C×H×W = (64×9 + 64×64)×112×112。比例 = (9+C)/(9C) = 73/576 ≈ 0.127。
